# BurnInWatch — Dataset Discovery & Initial EDA

This notebook performs **dataset discovery only**. It does not invent labels, alter measurements, or train a model yet.

Goals:
- inventory every file in the NASA IGBT dataset
- inspect `.mat` variables and dimensions
- inspect Excel workbooks and sheets
- identify likely time-series / device-level data
- export inventory CSVs for the next analysis stage

Run this first and share the important outputs before we decide preprocessing, feature engineering, train/test splitting, or the anomaly-detection model.

In [1]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

print("Environment ready")

Environment ready


In [3]:
# Change this only if your folder is somewhere else.
# Expected structure:
# BurnInWatch/
#   dataset/
#      Data/
#         ...

DATA_ROOT = Path("dataset/Data").resolve()

print("Dataset root:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())

if not DATA_ROOT.exists():
    print("\nERROR: Change DATA_ROOT to your actual Data folder.")

Dataset root: C:\Users\Abhishek Kuma\Desktop\BurnInWatch\dataset\Data
Exists: True


In [4]:
# ============================================================
# COMPLETE FILE INVENTORY
# ============================================================

rows = []

if DATA_ROOT.exists():
    for path in sorted(DATA_ROOT.rglob("*")):
        if path.is_file():
            rows.append({
                "relative_path": str(path.relative_to(DATA_ROOT)),
                "filename": path.name,
                "extension": path.suffix.lower(),
                "size_kb": round(path.stat().st_size / 1024, 2),
                "size_mb": round(path.stat().st_size / (1024**2), 4)
            })

file_inventory = pd.DataFrame(rows)

print("Total files:", len(file_inventory))
display(file_inventory)

if not file_inventory.empty:
    print("\nFiles by extension:")
    display(
        file_inventory.groupby("extension")
        .agg(files=("filename", "count"), total_mb=("size_mb", "sum"))
        .sort_values("files", ascending=False)
    )

    file_inventory.to_csv("dataset_file_inventory.csv", index=False)
    print("\nSaved: dataset_file_inventory.csv")

Total files: 177


,relative_path,filename,extension,size_kb,size_mb
0,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,11.txt,.txt,0.06,0.0001
1,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,Breakdown.csv,.csv,4.88,0.0048
2,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,LeakageIV.csv,.csv,4.88,0.0048
3,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,Turn On.csv,.csv,4.88,0.0048
4,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,12.txt,.txt,0.06,0.0001
5,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,Breakdown.csv,.csv,4.88,0.0048
6,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,LeakageIV.csv,.csv,4.88,0.0048
7,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,Turn On.csv,.csv,4.88,0.0048
8,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,13.txt,.txt,0.06,0.0001
9,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,Breakdown.csv,.csv,4.88,0.0048



Files by extension:


,files,total_mb
extension,,
.csv,120,0.5780
.txt,40,0.0040
.mat,12,243.2627
.doc,4,0.1318
.xls,1,0.0239



Saved: dataset_file_inventory.csv


In [5]:
# ============================================================
# DIRECTORY TREE
# ============================================================

def print_tree(root, max_depth=5):
    root = Path(root)
    print(root.name + "/")

    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.relative_to(root).parts)

        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "    " * depth

        for d in sorted(dirs):
            print(f"{indent}{d}/")

        for f in sorted(files):
            print(f"{indent}{f}")

if DATA_ROOT.exists():
    print_tree(DATA_ROOT)

Data/
SMU Data for new devices/
Thermal Overstress Aging with DC at gate/
Thermal Overstress Aging with Square Signal at gate/
Thermal Overstress Aging with Square Signal at gate and SMU data/
    IGBT-IRG4BC30K/
    MOSFET-IRF520Npbf/
    Readme.doc
        Part 11/
        Part 12/
        Part 13/
        Part 14/
        Part 15/
        Part 16/
        Part 17/
        Part 18/
        Part 19/
        Part 1A/
        Part 1B/
        Part 1C/
        Part 1D/
        Part 1E/
        Part 1J/
        Part 1K/
        Part 1L/
        Part 8J/
        Part 8K/
        Part 8L/
            11.txt
            Breakdown.csv
            LeakageIV.csv
            Turn On.csv
            12.txt
            Breakdown.csv
            LeakageIV.csv
            Turn On.csv
            13.txt
            Breakdown.csv
            LeakageIV.csv
            Turn On.csv
            14.txt
            Breakdown.csv
            LeakageIV.csv
            Turn On.csv
            15.txt
          

## MATLAB inspection

The first critical question is: **what signals are actually inside each `.mat` file?**

We will record variable names, shapes, data types, and basic numeric statistics where possible.

In [6]:
# ============================================================
# FIND MAT FILES
# ============================================================

mat_files = list(DATA_ROOT.rglob("*.mat")) if DATA_ROOT.exists() else []

print("MAT files found:", len(mat_files))
for p in mat_files:
    print(" -", p.relative_to(DATA_ROOT))

MAT files found: 12
 - Thermal Overstress Aging with DC at gate\20080429T135531.mat
 - Thermal Overstress Aging with Square Signal at gate\april22nd-23rdIgbtIRCG40BC30kd-A17.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 2\Device2  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 2\Device2b  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 2\Device2check  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 3\Device3  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 3\Device3b  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 3\Device3check  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 4\Device4  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 4\D

In [7]:
# ============================================================
# INSPECT MATLAB VARIABLES
# ============================================================

def describe_array(name, value):
    result = {
        "variable": name,
        "python_type": type(value).__name__,
        "shape": str(getattr(value, "shape", None)),
        "dtype": str(getattr(value, "dtype", None)),
        "numeric": False,
        "n_elements": None,
        "min": None,
        "max": None,
        "mean": None,
        "std": None,
    }

    try:
        arr = np.asarray(value)
        result["n_elements"] = arr.size

        if np.issubdtype(arr.dtype, np.number):
            finite = arr[np.isfinite(arr)]
            result["numeric"] = True

            if finite.size:
                result["min"] = float(np.min(finite))
                result["max"] = float(np.max(finite))
                result["mean"] = float(np.mean(finite))
                result["std"] = float(np.std(finite))
    except Exception:
        pass

    return result


try:
    from scipy.io import loadmat
except ImportError:
    loadmat = None
    print("scipy is missing. Install it with: pip install scipy")


mat_inventory = []

if loadmat is not None:
    for path in mat_files:
        print("\n" + "=" * 80)
        print(path.relative_to(DATA_ROOT))
        print("=" * 80)

        try:
            data = loadmat(path, squeeze_me=False, struct_as_record=False)
            public_keys = [k for k in data.keys() if not k.startswith("__")]

            print("Variables:", public_keys)

            for key in public_keys:
                info = describe_array(key, data[key])
                info["file"] = str(path.relative_to(DATA_ROOT))
                mat_inventory.append(info)

                print(
                    f"{key:40s} "
                    f"shape={info['shape']:20s} "
                    f"dtype={info['dtype']}"
                )

        except Exception as e:
            print("Could not read with scipy.io.loadmat:", repr(e))
            print("If this is MATLAB v7.3, we will inspect it with HDF5/h5py next.")


mat_inventory_df = pd.DataFrame(mat_inventory)

if not mat_inventory_df.empty:
    display(mat_inventory_df)
    mat_inventory_df.to_csv("mat_variable_inventory.csv", index=False)
    print("Saved: mat_variable_inventory.csv")


Thermal Overstress Aging with DC at gate\20080429T135531.mat
Variables: ['index', 'x', 'gate_current', 'time', 'collector_current', 'heat_sink', 'package_temperture', 'gate_voltage', 'collector_voltage', 'GATE_VOLTAGE', 'COLLECTOR_VOLTAGE', 'HEAT_SINK_TEMP', 'GATE_CURRENT', 'COLLECTOR_CURRENT', 'TIME', 'PACKAGE_TEMP', 'current_time', 'current_date', 'data_path', 'comment']
index                                    shape=(1, 1)               dtype=int32
x                                        shape=(1,)                 dtype=<U18
gate_current                             shape=(1, 1)               dtype=float64
time                                     shape=(1, 1)               dtype=float64
collector_current                        shape=(1, 1)               dtype=float64
heat_sink                                shape=(1, 1)               dtype=float64
package_temperture                       shape=(1, 1)               dtype=float64
gate_voltage                             shape=(1, 1) 

,variable,python_type,shape,dtype,numeric,n_elements,min,max,mean,std,file
0,index,ndarray,"(1, 1)",int32,True,1,3.016810e+05,3.016810e+05,3.016810e+05,0.000000,Thermal Overstress Aging with DC at gate\20080...
1,x,ndarray,"(1,)",<U18,False,1,NaN,NaN,NaN,NaN,Thermal Overstress Aging with DC at gate\20080...
2,gate_current,ndarray,"(1, 1)",float64,True,1,3.099858e-04,3.099858e-04,3.099858e-04,0.000000,Thermal Overstress Aging with DC at gate\20080...
3,time,ndarray,"(1, 1)",float64,True,1,3.292347e+09,3.292347e+09,3.292347e+09,0.000000,Thermal Overstress Aging with DC at gate\20080...
4,collector_current,ndarray,"(1, 1)",float64,True,1,7.894338e+00,7.894338e+00,7.894338e+00,0.000000,Thermal Overstress Aging with DC at gate\20080...
5,heat_sink,ndarray,"(1, 1)",float64,True,1,1.714580e-01,1.714580e-01,1.714580e-01,0.000000,Thermal Overstress Aging with DC at gate\20080...
6,package_temperture,ndarray,"(1, 1)",float64,True,1,3.051656e+02,3.051656e+02,3.051656e+02,0.000000,Thermal Overstress Aging with DC at gate\20080...
7,gate_voltage,ndarray,"(1, 1)",float64,True,1,-1.644894e-02,-1.644894e-02,-1.644894e-02,0.000000,Thermal Overstress Aging with DC at gate\20080...
8,collector_voltage,ndarray,"(1, 1)",float64,True,1,3.000472e+00,3.000472e+00,3.000472e+00,0.000000,Thermal Overstress Aging with DC at gate\20080...
9,GATE_VOLTAGE,ndarray,"(1, 301680)",float64,True,301680,-8.474687e-02,2.729620e+00,1.355372e+00,1.332899,Thermal Overstress Aging with DC at gate\20080...


Saved: mat_variable_inventory.csv


In [8]:
# ============================================================
# NUMERIC MATLAB VARIABLE SUMMARY
# ============================================================

if not mat_inventory_df.empty:
    numeric_vars = mat_inventory_df[
        mat_inventory_df["numeric"] == True
    ].copy()

    display(
        numeric_vars[
            ["file", "variable", "shape", "n_elements",
             "min", "max", "mean", "std"]
        ]
    )
else:
    print("No MATLAB variables were successfully inspected.")

,file,variable,shape,n_elements,min,max,mean,std
0,Thermal Overstress Aging with DC at gate\20080...,index,"(1, 1)",1,3.016810e+05,3.016810e+05,3.016810e+05,0.000000
2,Thermal Overstress Aging with DC at gate\20080...,gate_current,"(1, 1)",1,3.099858e-04,3.099858e-04,3.099858e-04,0.000000
3,Thermal Overstress Aging with DC at gate\20080...,time,"(1, 1)",1,3.292347e+09,3.292347e+09,3.292347e+09,0.000000
4,Thermal Overstress Aging with DC at gate\20080...,collector_current,"(1, 1)",1,7.894338e+00,7.894338e+00,7.894338e+00,0.000000
5,Thermal Overstress Aging with DC at gate\20080...,heat_sink,"(1, 1)",1,1.714580e-01,1.714580e-01,1.714580e-01,0.000000
6,Thermal Overstress Aging with DC at gate\20080...,package_temperture,"(1, 1)",1,3.051656e+02,3.051656e+02,3.051656e+02,0.000000
7,Thermal Overstress Aging with DC at gate\20080...,gate_voltage,"(1, 1)",1,-1.644894e-02,-1.644894e-02,-1.644894e-02,0.000000
8,Thermal Overstress Aging with DC at gate\20080...,collector_voltage,"(1, 1)",1,3.000472e+00,3.000472e+00,3.000472e+00,0.000000
9,Thermal Overstress Aging with DC at gate\20080...,GATE_VOLTAGE,"(1, 301680)",301680,-8.474687e-02,2.729620e+00,1.355372e+00,1.332899
10,Thermal Overstress Aging with DC at gate\20080...,COLLECTOR_VOLTAGE,"(1, 301680)",301680,1.583185e+00,9.988193e+00,5.728200e+00,3.980162


## Excel inspection

Excel files may contain device IDs, aging logs, operating conditions, timestamps, or parameter characterization data.

In [9]:
# ============================================================
# EXCEL WORKBOOK INSPECTION
# ============================================================

excel_files = []

for ext in ("*.xlsx", "*.xls", "*.xlsm"):
    excel_files.extend(DATA_ROOT.rglob(ext))

print("Excel files found:", len(excel_files))

for p in excel_files:
    print(" -", p.relative_to(DATA_ROOT))


excel_inventory = []

for path in excel_files:
    try:
        xl = pd.ExcelFile(path)

        print("\n" + "=" * 80)
        print(path.relative_to(DATA_ROOT))
        print("Sheets:", xl.sheet_names)

        for sheet in xl.sheet_names:
            try:
                df = pd.read_excel(path, sheet_name=sheet)

                excel_inventory.append({
                    "file": str(path.relative_to(DATA_ROOT)),
                    "sheet": sheet,
                    "rows": len(df),
                    "columns": len(df.columns),
                    "columns_list": " | ".join(map(str, df.columns))
                })

                print(f"\nSheet: {sheet}")
                print("Shape:", df.shape)
                print("Columns:", list(df.columns))
                display(df.head(5))

            except Exception as e:
                print(f"Could not read sheet {sheet}: {e}")

    except Exception as e:
        print("Could not open:", path, "|", repr(e))


excel_inventory_df = pd.DataFrame(excel_inventory)

if not excel_inventory_df.empty:
    display(excel_inventory_df)
    excel_inventory_df.to_csv("excel_sheet_inventory.csv", index=False)
    print("Saved: excel_sheet_inventory.csv")

Excel files found: 1
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Device Aging log.xls
Could not open: C:\Users\Abhishek Kuma\Desktop\BurnInWatch\dataset\Data\Thermal Overstress Aging with Square Signal at gate and SMU data\Device Aging log.xls | ImportError("Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.")


In [10]:
# ============================================================
# SEARCH FOR LIKELY IMPORTANT FIELDS
# ============================================================

keywords = [
    "time", "timestamp", "date", "cycle", "sample",
    "voltage", "volt", "vce", "vge", "gate",
    "current", "amp", "collector", "ic",
    "temperature", "temp", "thermal",
    "power", "resistance", "rds", "rce",
    "device", "igbt", "mosfet", "aging", "age", "failure"
]

def keyword_hits(text):
    text = str(text).lower()
    return [k for k in keywords if k in text]


if not excel_inventory_df.empty:
    all_columns = []

    for value in excel_inventory_df["columns_list"].dropna():
        all_columns.extend(str(value).split(" | "))

    all_columns = sorted(set(all_columns))

    hits = pd.DataFrame({
        "field": all_columns,
        "possible_keywords": [
            ", ".join(keyword_hits(c)) for c in all_columns
        ]
    })

    display(hits[hits["possible_keywords"] != ""])
else:
    print("No Excel column inventory available yet.")

No Excel column inventory available yet.


## What we deliberately do NOT do yet

- No fabricated data
- No synthetic anomalies
- No arbitrary labels
- No random row-wise train/test split
- No arbitrary frontend fields
- No model training yet

After this notebook, we will create the **canonical cleaned CSV** from the actual measurements. Then we will do feature engineering, leakage-safe splitting, and unsupervised anomaly detection.